# Tutorial 6: Strategy Benchmark and Comparison Guide

**How to choose the right continual learning strategy for your use case.**

This project provides five strategies for learning from new documents at inference time.
Each makes different trade-offs between accuracy, speed, forgetting, and hardware requirements.
This tutorial benchmarks all five on the same data so you can make an informed choice.

| Strategy | Approach | Weight Changes? |
|----------|---------|----------------|
| TTT-E2E | Gradient descent on document | Yes (trainable MLP) |
| JitRL MVP | TF-IDF retrieval + logit bias | No |
| JitRL Full | Hidden-state retrieval + reward modulation | No |
| Doc-to-LoRA | Hypernetwork-generated LoRA adapters | Yes (adapter injection) |
| ACE | LLM reflection loops + playbook | No (external LLM) |

In [ ]:
# Cell 2: Load sample data and model
import time
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32, trust_remote_code=True
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print(f"Model on {device}")

# Load sample document
document = Path("data/sample_document.txt").read_text()
print(f"Document: {len(document.split())} words")

# QA items for evaluation
qa_items = [
    {"question": "Who led the Quantum Computing Research Division?", "answer": "Dr. Sarah Chen"},
    {"question": "What year was the quantum division established?", "answer": "2019"},
    {"question": "What was the code name of the 72-qubit processor?", "answer": "RedShift"},
    {"question": "What quantum volume did RedShift achieve?", "answer": "512"},
    {"question": "What framework simulated quantum circuits?", "answer": "QubitFlow"},
    {"question": "What encryption protocol was developed with NIST?", "answer": "Lattice Shield"},
    {"question": "What was the division's annual budget?", "answer": "$120 million"},
    {"question": "How many researchers by end of 2024?", "answer": "ninety-five"},
]

# Holdout items for forgetting measurement (general knowledge, not from document)
holdout_items = [
    {"question": "What is the capital of France?", "answer": "Paris"},
    {"question": "What is 2 + 2?", "answer": "4"},
    {"question": "What planet is closest to the sun?", "answer": "Mercury"},
    {"question": "Who wrote Romeo and Juliet?", "answer": "Shakespeare"},
    {"question": "What is the chemical symbol for water?", "answer": "H2O"},
]

print(f"QA items: {len(qa_items)} (document), {len(holdout_items)} (holdout)")

## Forgetting Metrics Explained

The **forgetting ratio** measures how much existing knowledge is lost when learning new content:

```
forgetting_ratio = (accuracy_before - accuracy_after) / accuracy_before
```

| Ratio | Meaning |
|-------|--------|
| 0.0 | No forgetting at all |
| 0.0 - 0.1 | Minimal forgetting (acceptable) |
| 0.1 - 0.3 | Moderate forgetting (may need tuning) |
| 0.3 - 0.5 | Significant forgetting (problematic) |
| 0.5 - 1.0 | Severe catastrophic forgetting |
| < 0.0 | Negative = model actually improved! |

We measure this by testing the model on **holdout questions** (general knowledge unrelated
to the learned document) before and after each learning strategy.

In [ ]:
# Cell 4: Demo compute_forgetting_ratio with synthetic data
from continual_learning.evaluation.forgetting_metrics import compute_forgetting_ratio

# Synthetic examples
examples = [
    (0.80, 0.80, "No forgetting"),
    (0.80, 0.72, "Mild forgetting (10%)"),
    (0.80, 0.56, "Significant forgetting (30%)"),
    (0.80, 0.40, "Severe forgetting (50%)"),
    (0.80, 0.00, "Total catastrophic forgetting"),
    (0.80, 0.88, "Improvement (negative ratio)"),
    (0.00, 0.50, "Baseline was zero (edge case)"),
]

print(f"{'Before':>8} {'After':>8} {'Ratio':>8}  Description")
print("-" * 55)
for before, after, desc in examples:
    ratio = compute_forgetting_ratio(before, after)
    print(f"{before:>8.2f} {after:>8.2f} {ratio:>8.4f}  {desc}")

In [ ]:
# Cell 5: Benchmark TTT-E2E
# Note: TTT-E2E requires DualMLP-modified model. We simulate the benchmark
# structure here. On a full setup, use the modified Qwen model.
from continual_learning.evaluation.benchmarks import evaluate_qa_accuracy

results = {}  # Collect all benchmark results here

print("Benchmark: TTT-E2E")
print("=" * 50)
print("Note: TTT-E2E requires DualMLP injection (layers 21-28).")
print("This cell demonstrates the measurement pattern.")
print("For full TTT-E2E benchmarks, use scripts/validate_gpu.py\n")

# Measure baseline holdout accuracy
baseline_holdout = evaluate_qa_accuracy(model, tokenizer, holdout_items)
print(f"Baseline holdout accuracy: {baseline_holdout:.2%}")

# TTT-E2E would modify model weights here via gradient descent
# For this demo, we record a placeholder
results["TTT-E2E"] = {
    "accuracy": None,  # Requires DualMLP model
    "forgetting": None,
    "learn_time": None,
    "eval_time": None,
    "note": "Requires DualMLP-modified model (see scripts/validate_gpu.py)",
}
print("Skipped (requires DualMLP setup). Will be included in summary as N/A.")

In [ ]:
# Cell 6: Benchmark JitRL MVP
from continual_learning.jitrl.mvp.engine import JitRLMVPEngine

print("Benchmark: JitRL MVP")
print("=" * 50)

mvp_engine = JitRLMVPEngine(model=model, tokenizer=tokenizer)

# Learn
t0 = time.time()
mvp_engine.learn(document)
mvp_learn_time = time.time() - t0

# Evaluate on document QA
t0 = time.time()
mvp_correct = 0
for item in qa_items:
    response = mvp_engine.generate(f"Answer concisely: {item['question']}", max_new_tokens=80)
    if item["answer"].lower() in response.lower():
        mvp_correct += 1
mvp_eval_time = time.time() - t0
mvp_accuracy = mvp_correct / len(qa_items)

# Forgetting: MVP does not modify weights, so forgetting = 0
mvp_forgetting = 0.0

results["JitRL MVP"] = {
    "accuracy": mvp_accuracy,
    "forgetting": mvp_forgetting,
    "learn_time": mvp_learn_time,
    "eval_time": mvp_eval_time,
}

print(f"  Accuracy:   {mvp_accuracy:.2%} ({mvp_correct}/{len(qa_items)})")
print(f"  Forgetting: {mvp_forgetting:.4f} (no weight changes)")
print(f"  Learn time: {mvp_learn_time:.4f}s")
print(f"  Eval time:  {mvp_eval_time:.2f}s")

mvp_engine.clear()

In [ ]:
# Cell 7: Benchmark JitRL Full
from continual_learning.jitrl.full.engine import JitRLFullEngine

print("Benchmark: JitRL Full")
print("=" * 50)

full_engine = JitRLFullEngine(
    model=model, tokenizer=tokenizer, modulation_temperature=0.5
)

# Learn
t0 = time.time()
full_engine.learn(document)
full_learn_time = time.time() - t0

# Evaluate on document QA
t0 = time.time()
full_correct = 0
for item in qa_items:
    response = full_engine.generate(f"Answer concisely: {item['question']}", max_new_tokens=80)
    if item["answer"].lower() in response.lower():
        full_correct += 1
full_eval_time = time.time() - t0
full_accuracy = full_correct / len(qa_items)

# Forgetting: Full does not modify weights, so forgetting = 0
full_forgetting = 0.0

results["JitRL Full"] = {
    "accuracy": full_accuracy,
    "forgetting": full_forgetting,
    "learn_time": full_learn_time,
    "eval_time": full_eval_time,
}

print(f"  Accuracy:   {full_accuracy:.2%} ({full_correct}/{len(qa_items)})")
print(f"  Forgetting: {full_forgetting:.4f} (no weight changes)")
print(f"  Learn time: {full_learn_time:.4f}s")
print(f"  Eval time:  {full_eval_time:.2f}s")

full_engine.clear()

In [ ]:
# Cell 8: Benchmark Doc-to-LoRA
from continual_learning.doc2lora.engine import Doc2LoRAEngine

print("Benchmark: Doc-to-LoRA")
print("=" * 50)

# Measure baseline holdout accuracy before LoRA injection
baseline_holdout = evaluate_qa_accuracy(model, tokenizer, holdout_items)
print(f"  Baseline holdout: {baseline_holdout:.2%}")

d2l_engine = Doc2LoRAEngine(
    model=model, tokenizer=tokenizer,
    hidden_dim=2048, num_target_layers=4,
    intermediate_dim=512, lora_rank=8,
    mode="doc", simulated=True,
    layer_prefix="model.layers.{i}.mlp",
)

# Learn
t0 = time.time()
d2l_engine.learn(document)
d2l_learn_time = time.time() - t0

# Evaluate on document QA
t0 = time.time()
d2l_correct = 0
for item in qa_items:
    response = d2l_engine.generate(f"Answer concisely: {item['question']}", max_new_tokens=80)
    if item["answer"].lower() in response.lower():
        d2l_correct += 1
d2l_eval_time = time.time() - t0
d2l_accuracy = d2l_correct / len(qa_items)

# Measure forgetting on holdout
post_holdout = evaluate_qa_accuracy(model, tokenizer, holdout_items)
d2l_forgetting = compute_forgetting_ratio(baseline_holdout, post_holdout)

results["Doc-to-LoRA"] = {
    "accuracy": d2l_accuracy,
    "forgetting": d2l_forgetting,
    "learn_time": d2l_learn_time,
    "eval_time": d2l_eval_time,
}

print(f"  Accuracy:   {d2l_accuracy:.2%} ({d2l_correct}/{len(qa_items)})")
print(f"  Forgetting: {d2l_forgetting:.4f}")
print(f"  Learn time: {d2l_learn_time:.4f}s")
print(f"  Eval time:  {d2l_eval_time:.2f}s")

# Clean up — restore original weights
d2l_engine.clear()

In [ ]:
# Cell 9: Benchmark ACE (optional — requires Ollama)
import urllib.request
import json

print("Benchmark: ACE (optional)")
print("=" * 50)

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen3.5:9b"

try:
    req = urllib.request.Request(f"{OLLAMA_URL}/api/tags")
    with urllib.request.urlopen(req, timeout=3) as resp:
        pass
    ollama_available = True
except Exception:
    ollama_available = False

if ollama_available:
    from continual_learning.ace.engine import ACEEngine

    ace_engine = ACEEngine(
        ollama_model=OLLAMA_MODEL,
        ollama_base_url=OLLAMA_URL,
        num_loops=2,
        max_strategies=50,
    )

    ace_qa = [
        {"question": "Who led the quantum division?", "answer": "Dr. Sarah Chen"},
        {"question": "What was RedShift?", "answer": "72-qubit trapped-ion processor"},
    ]

    t0 = time.time()
    ace_engine.learn(document, qa_pairs=ace_qa)
    ace_learn_time = time.time() - t0

    t0 = time.time()
    ace_correct = 0
    for item in qa_items:
        response = ace_engine.generate(item["question"])
        if item["answer"].lower() in response.lower():
            ace_correct += 1
    ace_eval_time = time.time() - t0
    ace_accuracy = ace_correct / len(qa_items)

    results["ACE"] = {
        "accuracy": ace_accuracy,
        "forgetting": 0.0,  # ACE never modifies the model
        "learn_time": ace_learn_time,
        "eval_time": ace_eval_time,
    }

    print(f"  Accuracy:   {ace_accuracy:.2%} ({ace_correct}/{len(qa_items)})")
    print(f"  Forgetting: 0.0000 (no weight changes)")
    print(f"  Learn time: {ace_learn_time:.2f}s")
    print(f"  Eval time:  {ace_eval_time:.2f}s")
    ace_engine.clear()
else:
    results["ACE"] = {
        "accuracy": None,
        "forgetting": None,
        "learn_time": None,
        "eval_time": None,
        "note": "Ollama not available",
    }
    print("  Skipped: Ollama not running. Start with 'ollama serve' to include ACE.")

In [ ]:
# Cell 10: Results comparison table
print("\n" + "=" * 80)
print("BENCHMARK RESULTS COMPARISON")
print("=" * 80)

header = f"{'Strategy':<15} {'Accuracy':>10} {'Forgetting':>12} {'Learn (s)':>10} {'Eval (s)':>10}"
print(header)
print("-" * len(header))

for name, r in results.items():
    acc = f"{r['accuracy']:.2%}" if r["accuracy"] is not None else "N/A"
    fgt = f"{r['forgetting']:.4f}" if r["forgetting"] is not None else "N/A"
    lt = f"{r['learn_time']:.4f}" if r["learn_time"] is not None else "N/A"
    et = f"{r['eval_time']:.2f}" if r["eval_time"] is not None else "N/A"
    print(f"{name:<15} {acc:>10} {fgt:>12} {lt:>10} {et:>10}")

print("\nNotes:")
print("- TTT-E2E requires DualMLP model setup (see scripts/validate_gpu.py)")
print("- ACE requires Ollama running locally")
print("- Forgetting = 0 for retrieval-based methods (no weight changes)")
print("- Doc-to-LoRA uses SimulatedHypernetwork (real checkpoint may differ)")

In [ ]:
# Cell 11: Matplotlib bar charts comparing strategies
try:
    import matplotlib
    matplotlib.use("Agg")  # Non-interactive backend for notebooks
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("matplotlib not installed. Install with: pip install matplotlib")
    print("Skipping visualizations.")

if HAS_MATPLOTLIB:
    # Filter to strategies with actual results
    plot_data = {k: v for k, v in results.items() if v["accuracy"] is not None}

    if plot_data:
        names = list(plot_data.keys())
        accuracies = [plot_data[n]["accuracy"] for n in names]
        forgettings = [plot_data[n]["forgetting"] for n in names]
        learn_times = [plot_data[n]["learn_time"] for n in names]

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Accuracy bar chart
        colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
        axes[0].bar(names, accuracies, color=colors[:len(names)])
        axes[0].set_ylabel("Accuracy")
        axes[0].set_title("Document QA Accuracy")
        axes[0].set_ylim(0, 1.0)
        for i, v in enumerate(accuracies):
            axes[0].text(i, v + 0.02, f"{v:.0%}", ha="center", fontsize=10)

        # Forgetting bar chart
        axes[1].bar(names, forgettings, color=colors[:len(names)])
        axes[1].set_ylabel("Forgetting Ratio")
        axes[1].set_title("Catastrophic Forgetting")
        axes[1].set_ylim(-0.1, max(max(forgettings) + 0.1, 0.5))
        axes[1].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
        for i, v in enumerate(forgettings):
            axes[1].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)

        # Learn time bar chart (log scale)
        axes[2].bar(names, learn_times, color=colors[:len(names)])
        axes[2].set_ylabel("Time (seconds)")
        axes[2].set_title("Learning Time")
        axes[2].set_yscale("log")
        for i, v in enumerate(learn_times):
            axes[2].text(i, v * 1.3, f"{v:.3f}s", ha="center", fontsize=9)

        for ax in axes:
            ax.tick_params(axis="x", rotation=30)

        plt.tight_layout()
        plt.savefig("benchmark_results.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Chart saved to benchmark_results.png")
    else:
        print("No strategies with results to plot.")

## Decision Guide: Choosing the Right Strategy

Use this flowchart to pick the best approach for your use case:

```
START
  |
  +-- Can you modify the model weights?
  |     |
  |     +-- NO --> Can you run an LLM server (Ollama)?
  |     |           |
  |     |           +-- YES --> ACE (playbook-based, zero-weight)
  |     |           +-- NO  --> JitRL MVP (TF-IDF retrieval, instant)
  |     |
  |     +-- YES --> Is learning speed critical?
  |                   |
  |                   +-- YES --> Doc-to-LoRA (single forward pass)
  |                   +-- NO  --> Do you need deep knowledge retention?
  |                                 |
  |                                 +-- YES --> TTT-E2E (gradient descent)
  |                                 +-- NO  --> JitRL Full (semantic retrieval)
```

**Quick reference:**

| Need | Best Strategy |
|------|------|
| Fastest possible learning | JitRL MVP |
| Best accuracy on paraphrased queries | JitRL Full |
| No weight changes + inspectable knowledge | ACE |
| Fast weight adaptation (single pass) | Doc-to-LoRA |
| Deepest knowledge internalization | TTT-E2E |
| Minimal hardware requirements | JitRL MVP or ACE |
| Production deployment (API-only model) | ACE or JitRL MVP |

## Hardware Considerations

| Strategy | GPU Required? | VRAM (Qwen2.5-1.5B) | CPU Feasible? |
|----------|:------------:|:-------------------:|:------------:|
| TTT-E2E | Yes (training) | ~6-8 GB | Very slow |
| JitRL MVP | For generation | ~4 GB | Yes (slow generation) |
| JitRL Full | Yes (hidden states) | ~5-6 GB | Slow but possible |
| Doc-to-LoRA | For generation | ~4-5 GB | Yes (slow generation) |
| ACE | No (uses Ollama) | Ollama handles it | Yes |

**Scaling to larger models (7B, 13B, 70B):**
- JitRL MVP and ACE scale best — they add minimal overhead to the base model
- TTT-E2E and Doc-to-LoRA scale linearly with model size
- JitRL Full's Knowledge Store is model-size-dependent (hidden dimension)

**Multi-GPU considerations:**
- All strategies work with model parallelism (device_map="auto")
- TTT-E2E gradient computation may need careful memory management on split models

## Exercises

1. **Multi-document scaling**: Add 2, 5, and 10 documents to each strategy.
   Plot accuracy and learn time vs. number of documents. Which strategies scale best?

2. **Question difficulty**: Create easy, medium, and hard question sets.
   Easy = exact quotes from the document. Hard = requires synthesis across paragraphs.
   Which strategies handle hard questions best?

3. **Forgetting under load**: For weight-modifying strategies (TTT-E2E, Doc-to-LoRA),
   learn 5 documents sequentially and measure forgetting after each. Does forgetting
   accumulate? Is there a tipping point?

4. **Ensemble approach**: Combine two strategies — for example, use JitRL MVP for
   context retrieval and Doc-to-LoRA for knowledge internalization. Does the combination
   outperform either strategy alone?

5. **Latency profiling**: Break down each strategy's generate() call into components
   (retrieval time, modulation time, actual generation time). Where are the bottlenecks?